In [108]:
%reload_ext autoreload
%autoreload 2

In [109]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [110]:
from solarrpy import seasonalClearsky
from solarrpy import seasonalModel

In [111]:
"""
import cdsapi

dataset = "cams-solar-radiation-timeseries"
request = {
    "sky_type": "observed_cloud",
    "location": {"longitude": 11.3426, "latitude": 44.4949},
    "altitude": ["71"],
    "date": ["2005-01-01/2026-03-31"],
    "time_step": "1day",
    "time_reference": "true_solar_time",
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()
"""

'\nimport cdsapi\n\ndataset = "cams-solar-radiation-timeseries"\nrequest = {\n    "sky_type": "observed_cloud",\n    "location": {"longitude": 11.3426, "latitude": 44.4949},\n    "altitude": ["71"],\n    "date": ["2005-01-01/2026-03-31"],\n    "time_step": "1day",\n    "time_reference": "true_solar_time",\n    "data_format": "csv"\n}\n\nclient = cdsapi.Client()\nclient.retrieve(dataset, request).download()\n'

In [112]:
df = pd.read_csv("../data/Bologna.csv")
spec = {
    'target': 'GHI',
    'coords': {
        "lat": 44.4949,
        "lon": 11.3426,
        "alt": 71
    },
    'data': df
}

In [123]:
# ----------------------------------------------------------------------------
# NOTE: This script assumes the existence of the previously translated classes/functions:
# SeasonalClearsky, control_seasonalClearsky, clearsky_outliers
# 
# It also assumes a pre-existing `spec` object/dictionary that contains the dataset.
# For example:
# spec = {
#     'data': pd.DataFrame({'GHI': [...], 'date': [...], 'clearsky': [...], 'n': [...]}),
#     'coords': {'lat': 45.0},
#     'target': 'GHI'
# }
# ----------------------------------------------------------------------------

# ==========================================
# Inputs
# ==========================================

# Control parameters
control = seasonalClearsky.control_seasonalClearsky(
    orders=1, order_H0=1, periods=365, 
    include_intercept=True, include_trend=False,
    delta0=1.4, lower=0, upper=3, by=0.001, ntol=10, quiet=False
)

# Extracting from `spec` object
data_all = spec['data'].copy()
data_all['date'] = pd.to_datetime(data_all['date'])
lat = spec['coords']['lat']
alt = spec['coords']['alt']
target_col = spec['target']

# Command parameters
plot_data = False
test_outputs = False

all_params = []

for year_i in range(2013, 2023):
    mask = data_all['date'].dt.year <= year_i
    data = data_all.loc[mask].copy()
    
    model_coefficients = {}

    print(f"\nRunning model with data up to {year_i}-12-31 ({len(data)} rows)")

    GHI = data['GHI']
    date = data['date']
    clearsky = data['clearsky']
    H0 = data['H0']

    # ==========================================
    # Fit the clear sky model
    # ==========================================

    # Initialize the model 
    clearsky_model = seasonalClearsky.SeasonalClearsky(control=control)

    # Fit the parameters
    clearsky_model.fit(GHI, date, lat, clearsky, alt=alt)
    #print(clearsky_model)
    
    # Predictions
    data['Ct'] = clearsky_model.predict(n=data['n'], newdata=data)

    # Save the model coefficients
    delta0, delta1, delta2, delta3 = clearsky_model._model.params.values[:4]
    delta0_err, delta1_err, delta2_err, delta3_err = clearsky_model._model.bse.values[:4]

    #Computing alpha and beta
    eps = 1e-3 * min(1 - GHI / data['Ct'])
    alpha_t = min(1 - GHI / data['Ct']) - eps
    beta_t  = max(1 - GHI / data['Ct']) - min(1 - GHI / data['Ct']) + 2 * eps
    
    # Computing Ybar
    Ybar = np.log(np.log(beta_t) - np.log(1 - alpha_t - GHI / data['Ct']))
    Ybar.fillna(np.mean(Ybar), inplace=True)
    data['Ybar'] = Ybar

    seasonal_model = seasonalModel.SeasonalModel(orders=[1], periods=[365])
    seasonal_model.fit(data=data[['Ybar', 'n']], target_col='Ybar', time_col='n', include_intercept=True)

    # 6. Extract seasonal parameters and update the dictionary all at once
    a0, a1, a2 = seasonal_model._model.params.values[:3]
    a0_err, a1_err, a2_err = seasonal_model._std_errors.values[:3]
    
    model_coefficients.update({
        'Year': year_i,
        'alpha': alpha_t,
        'beta': beta_t,
        'delta0': delta0,
        'delta1': delta1,
        'delta2': delta2,
        'delta3': delta3,
        'delta0_err': delta0_err,
        'delta1_err': delta1_err,
        'delta2_err': delta2_err,
        'delta3_err': delta3_err,
        'a0': a0,
        'a1': a1,
        'a2': a2,
        'a0_err': a0_err,
        'a1_err': a1_err,
        'a2_err': a2_err
    })
    
    all_params.append(model_coefficients)

    # ==========================================
    # Plotting
    # ==========================================
    if plot_data:
        # Filter data between dates
        df_plot = data

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Plot 1: CAMS vs GHI
        axes[0].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[0].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[0].set_xlabel('Day of the year')
        axes[0].set_ylabel('Clear sky')
        axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[0].grid(True, linestyle='--', alpha=0.7)

        # Plot 2: Fitted vs CAMS
        axes[1].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[1].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[1].set_xlabel('Day of the year')
        axes[1].set_ylabel('Clear sky')
        axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[1].grid(True, linestyle='--', alpha=0.7)

        # Plot 3: Fitted vs GHI
        axes[2].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[2].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[2].set_xlabel('Day of the year')
        axes[2].set_ylabel('Clear sky')
        axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[2].grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()

    # ==========================================
    # Test: imputed outliers
    # ==========================================
    if test_outputs:
        # Impute outliers
        outliers = seasonalClearsky.clearsky_outliers(data[target_col], data['Ct'], data['date'], quiet=True)
        data[target_col] = outliers['x']
        
        # Test tolerance parameter
        print("\033[1;35m---------------\033[0m \033[1;32m  Test clearskyModel_control and clearskyModel_fit \033[1;35m---------------\033[0m")
        
        passed_ntol = outliers['n'] <= control['ntol']
        msg_ntol = "\033[1;32mPassed\033[0m!\n" if passed_ntol else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the number of outliers imputed is below {control['ntol']}...({outliers['n']}) {msg_ntol}")
        
        # ==========================================
        # Test: delta parameter
        # ==========================================
        
        # Test delta parameter (Accessing mangled private attribute)
        delta = clearsky_model.delta    
        test_delta = (delta > control['lower']) and (delta < control['upper'])
        msg_delta = "\033[1;32mPassed\033[0m!\n" if test_delta else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the parameter delta is inside lower ({control['lower']}) and upper ({control['upper']})...({delta}) {msg_delta}")
        
        # ==========================================
        # Test: order of seasonal components
        # ==========================================
        
        # Count the number of parameters 
        n_params_target = 1 if control['include_intercept'] else 0
        n_params_target += 1 if control['include_trend'] else 0
        n_params_target += control['orders'] * 2
        n_params_target += control['order_H0']
        
        # Test if the number of parameters is correct 
        n_params = len(clearsky_model._model.params)
        
        # Print result 
        msg_params = "\033[1;32mPassed\033[0m!\n" if n_params == n_params_target else "\033[1;31mNOT passed\033[0m! \n"
        print(f"Check if the number of parameters is equal to {n_params_target}...({n_params}) {msg_params}")
        
        # ==========================================
        # Differential
        # ==========================================
        
        # Note differential when a trend is true is not implemented
        dt = 0.05
        n0 = 34

        num_diff = (clearsky_model.predict(n=n0 + dt) - clearsky_model.predict(n=n0)) / dt
        ana_diff = clearsky_model.differential(n=n0)
        
        print(f"Numerical Differential: \n{num_diff}")
        print(f"Analytical Differential: \n{ana_diff}")


Running model with data up to 2013-12-31 (3287 rows)

Running model with data up to 2014-12-31 (3652 rows)


/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



Running model with data up to 2015-12-31 (4017 rows)

Running model with data up to 2016-12-31 (4383 rows)


/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



Running model with data up to 2017-12-31 (4748 rows)

Running model with data up to 2018-12-31 (5113 rows)


/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



Running model with data up to 2019-12-31 (5478 rows)

Running model with data up to 2020-12-31 (5844 rows)


/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



Running model with data up to 2021-12-31 (6209 rows)

Running model with data up to 2022-12-31 (6574 rows)


/opt/anaconda3/envs/finance_ml/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [124]:
params_df = pd.DataFrame(all_params)

# 6. Reorder the columns so 'Year' is the very first column
cols = ['Year'] + [col for col in params_df.columns if col != 'Year']
params_df = params_df[cols]

params_df

,Year,alpha,beta,delta0,delta1,delta2,delta3,delta0_err,delta1_err,delta2_err,delta3_err,a0,a1,a2,a0_err,a1_err,a2_err
0,2013,-0.023879,0.943665,-1.139138,0.920037,0.010834,0.495107,0.319647,0.042880,0.031798,0.182858,-0.066251,-0.068593,-0.392962,0.016333,0.023105,0.023091
1,2014,-0.022637,0.942537,-1.050383,0.909217,0.024340,0.437453,0.302052,0.040519,0.030048,0.172792,-0.074949,-0.063666,-0.388748,0.015463,0.021874,0.021862
2,2015,-0.023424,0.943276,-1.272883,0.938459,0.000474,0.567832,0.285686,0.038324,0.028420,0.163429,-0.066070,-0.056716,-0.377449,0.014722,0.020825,0.020814
3,2016,-0.016843,0.937216,-1.209786,0.935381,0.006032,0.528905,0.275402,0.036945,0.027397,0.157548,-0.069131,-0.062445,-0.375747,0.014132,0.019992,0.019979
4,2017,-0.015900,0.936332,-1.130120,0.924475,0.018619,0.484967,0.264013,0.035417,0.026264,0.151032,-0.049130,-0.060300,-0.372100,0.013527,0.019137,0.019124
5,2018,-0.014003,0.934568,-1.104480,0.921898,0.021886,0.473866,0.255501,0.034275,0.025417,0.146163,-0.051173,-0.073368,-0.379362,0.013049,0.018460,0.018449
6,2019,-0.011436,0.932247,-1.273501,0.945250,0.012313,0.576663,0.247172,0.033157,0.024588,0.141397,-0.039014,-0.068861,-0.365504,0.012623,0.017857,0.017847
7,2020,-0.011690,0.932520,-1.294752,0.948079,0.011371,0.589114,0.238579,0.032005,0.023734,0.136483,-0.027453,-0.060292,-0.357881,0.012192,0.017248,0.017236
8,2021,-0.007716,0.929038,-1.470046,0.975617,-0.000796,0.689025,0.232113,0.031137,0.023091,0.132784,-0.027774,-0.055509,-0.355725,0.011750,0.016622,0.016612
9,2022,-0.007954,0.929324,-1.635525,0.998165,-0.016517,0.782922,0.226603,0.030398,0.022542,0.129631,-0.020747,-0.049861,-0.353128,0.011375,0.016092,0.016082


In [125]:
params = params_df

# 1. Format the 'Train years' column
params['Train years'] = '2005-' + params['Year'].astype(str)

# 2. Add 'Obs.' (using the values from your screenshot)
params['Obs.'] = [3287, 3652, 4017, 4383, 4748, 5113, 5478, 5844, 6209, 6574]

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
params['a_0_fmt'] = params.apply(lambda row: f"{row['a0']:.4f}<br>({row['a0_err']:.4f})", axis=1)
params['a_1_fmt'] = params.apply(lambda row: f"{row['a1']:.4f}<br>({row['a1_err']:.4f})", axis=1)
params['a_2_fmt'] = params.apply(lambda row: f"{row['a2']:.4f}<br>({row['a2_err']:.4f})", axis=1)

# 3. Format the combined columns (Value + Standard Error with HTML <br>)
params['delta_0_fmt'] = params.apply(lambda row: f"{row['delta0']:.4f}<br>({row['delta0_err']:.4f})", axis=1)
params['delta_1_fmt'] = params.apply(lambda row: f"{row['delta1']:.4f}<br>({row['delta1_err']:.4f})", axis=1)
params['delta_2_fmt'] = params.apply(lambda row: f"{row['delta2']:.4f}<br>({row['delta2_err']:.4f})", axis=1)
params['delta_3_fmt'] = params.apply(lambda row: f"{row['delta3']:.4f}<br>({row['delta3_err']:.4f})", axis=1)

# Format the remaining columns to standard decimal lengths
params['alpha_fmt'] = params['alpha'].apply(lambda x: f"{x:.6f}")
params['beta_fmt'] = params['beta'].apply(lambda x: f"{x:.3f}")

# 4. Select and rename columns for the final display
display_df = params[['Train years', 'Obs.', 'alpha_fmt', 'beta_fmt', 
                     'delta_0_fmt', 'delta_1_fmt', 'delta_2_fmt', 'delta_3_fmt', 
                     'a_0_fmt', 'a_1_fmt', 'a_2_fmt']]

display_df.columns = ['Train years', 'Obs.', r'$\alpha$', r'$\beta$', 
                      r'$\delta_0$', r'$\delta_1$', r'$\delta_2$', r'$\delta_3$', 
                      r'$a_0$', r'$a_1$', r'$a_2$']

# Apply CSS Styling to mimic the LaTeX academic look
styles = [
    # Top and bottom thick horizontal rules
    {'selector': 'thead th', 'props': 'border-bottom: 1px solid black; border-top: 2px solid black; text-align: center; padding: 8px;'},
    {'selector': 'tbody tr:last-child td', 'props': 'border-bottom: 2px solid black;'},
    
    # Center all text and add vertical padding
    {'selector': 'tbody td', 'props': 'text-align: center; vertical-align: middle; padding: 8px;'},
    
    # Left-align the first column (Train years)
    {'selector': 'tbody td:nth-child(1)', 'props': 'text-align: left;'},
    
    # Selective vertical lines (matching the screenshot)
    # Column 3 is alpha, Column 5 is delta_0, Column 9 is a_0
    {'selector': 'th:nth-child(3), td:nth-child(3)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(5), td:nth-child(5)', 'props': 'border-left: 1px solid black;'},
    {'selector': 'th:nth-child(9), td:nth-child(9)', 'props': 'border-left: 1px solid black;'}
]

# Hide the default index and render
styled_table = display_df.style.set_table_styles(styles).hide(axis='index')
styled_table

Train years,Obs.,$\alpha$,$\beta$,$\delta_0$,$\delta_1$,$\delta_2$,$\delta_3$,$a_0$,$a_1$,$a_2$
2005-2013,3287,-0.023879,0.944,-1.1391(0.3196),0.9200(0.0429),0.0108(0.0318),0.4951(0.1829),-0.0663(0.0163),-0.0686(0.0231),-0.3930(0.0231)
2005-2014,3652,-0.022637,0.943,-1.0504(0.3021),0.9092(0.0405),0.0243(0.0300),0.4375(0.1728),-0.0749(0.0155),-0.0637(0.0219),-0.3887(0.0219)
2005-2015,4017,-0.023424,0.943,-1.2729(0.2857),0.9385(0.0383),0.0005(0.0284),0.5678(0.1634),-0.0661(0.0147),-0.0567(0.0208),-0.3774(0.0208)
2005-2016,4383,-0.016843,0.937,-1.2098(0.2754),0.9354(0.0369),0.0060(0.0274),0.5289(0.1575),-0.0691(0.0141),-0.0624(0.0200),-0.3757(0.0200)
2005-2017,4748,-0.015900,0.936,-1.1301(0.2640),0.9245(0.0354),0.0186(0.0263),0.4850(0.1510),-0.0491(0.0135),-0.0603(0.0191),-0.3721(0.0191)
2005-2018,5113,-0.014003,0.935,-1.1045(0.2555),0.9219(0.0343),0.0219(0.0254),0.4739(0.1462),-0.0512(0.0130),-0.0734(0.0185),-0.3794(0.0184)
2005-2019,5478,-0.011436,0.932,-1.2735(0.2472),0.9453(0.0332),0.0123(0.0246),0.5767(0.1414),-0.0390(0.0126),-0.0689(0.0179),-0.3655(0.0178)
2005-2020,5844,-0.011690,0.933,-1.2948(0.2386),0.9481(0.0320),0.0114(0.0237),0.5891(0.1365),-0.0275(0.0122),-0.0603(0.0172),-0.3579(0.0172)
2005-2021,6209,-0.007716,0.929,-1.4700(0.2321),0.9756(0.0311),-0.0008(0.0231),0.6890(0.1328),-0.0278(0.0118),-0.0555(0.0166),-0.3557(0.0166)
2005-2022,6574,-0.007954,0.929,-1.6355(0.2266),0.9982(0.0304),-0.0165(0.0225),0.7829(0.1296),-0.0207(0.0114),-0.0499(0.0161),-0.3531(0.0161)


# Computing the Seasonal Mean and Variance

In [ ]:
for year_i in range(2013, 2023):
    params = params_df[params_df['Year'] == year_i]
inner = np.log(beta) - np.log(1 - alpha - R / C)
Y = np.log(inner)

NameError: name 'beta' is not defined